<a href="https://colab.research.google.com/github/hiteshraos/AI-LAB/blob/main/WEEK6_1BM24CS117_PROPOSITIONAL_LOGIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import re
from itertools import product


class Expr:
    """A node in a propositional logic expression tree."""

    def __init__(self, op, *args):
        self.op = op
        self.args = args

    def __repr__(self):
        if not self.args:
            return str(self.op)
        if self.op == "~":
            return f"(~{self.args[0]})"
        return f"({self.args[0]} {self.op} {self.args[1]})"

    def evaluate(self, model):
        if not self.args:
            return model[self.op]

        args_eval = [arg.evaluate(model) for arg in self.args]

        if self.op == "~":
            return not args_eval[0]
        if self.op == "&":
            return args_eval[0] and args_eval[1]
        if self.op == "|":
            return args_eval[0] or args_eval[1]
        if self.op == ">>":
            return (not args_eval[0]) or args_eval[1]
        if self.op == "<<>>":
            return args_eval[0] == args_eval[1]

        raise ValueError(f"Unknown operator: {self.op}")

    def get_symbols(self):
        if not self.args:
            return {self.op}

        symbols = set()
        for arg in self.args:
            symbols.update(arg.get_symbols())
        return symbols


def parse_expr(expr_str):
    """Convert a propositional formula into an expression tree."""

    s = expr_str.strip()

    # Convert custom operators into Python-compatible operators.
    s = s.replace("<<>>", "==")
    s = s.replace(">>", "<=")

    tokens = re.findall(r"[a-zA-Z_][a-zA-Z0-9_]*", s)

    if not tokens:
        raise ValueError(
            f"No propositional symbols found in: '{expr_str}'"
        )

    namespace = {t: Expr(t) for t in tokens}

    class LogicWrapper:
        def __init__(self, obj):
            self.obj = obj

        def __invert__(self):
            return LogicWrapper(Expr("~", self.obj))

        def __or__(self, other):
            return LogicWrapper(Expr("|", self.obj, other.obj))

        def __and__(self, other):
            return LogicWrapper(Expr("&", self.obj, other.obj))

        def __le__(self, other):
            return LogicWrapper(Expr(">>", self.obj, other.obj))

        def __eq__(self, other):
            return LogicWrapper(
                Expr("<<>>", self.obj, other.obj)
            )

    wrapped_ns = {
        k: LogicWrapper(v) for k, v in namespace.items()
    }

    s_wrapped = re.sub(
        r"([a-zA-Z_][a-zA-Z0-9_]*)",
        r'wrapped_ns["\1"]',
        s
    )

    try:
        result = eval(s_wrapped, {"wrapped_ns": wrapped_ns})
        return result.obj
    except Exception as e:
        raise ValueError(
            f"Invalid syntax in expression '{expr_str}': {e}"
        ) from e


def PL_TRUE(sentence, model):
    """Evaluate a formula or a list of formulas."""

    if isinstance(sentence, list):
        return all(s.evaluate(model) for s in sentence)

    return sentence.evaluate(model)


truth_table_rows = []


def TT_CHECK_ALL(KB, alpha, symbols, model):
    """Enumerate every possible truth assignment."""

    if not symbols:
        kb_val = PL_TRUE(KB, model)
        alpha_val = PL_TRUE(alpha, model)

        truth_table_rows.append({
            "model": model.copy(),
            "kb": kb_val,
            "alpha": alpha_val
        })

        # If KB is true, alpha must also be true.
        return (not kb_val) or alpha_val

    P = symbols[0]
    rest = symbols[1:]

    model_false = model.copy()
    model_false[P] = False

    model_true = model.copy()
    model_true[P] = True

    res_false = TT_CHECK_ALL(
        KB, alpha, rest, model_false
    )
    res_true = TT_CHECK_ALL(
        KB, alpha, rest, model_true
    )

    return res_false and res_true


def TT_ENTAILS(KB, alpha):
    """Check whether KB logically entails alpha."""

    global truth_table_rows
    truth_table_rows = []

    KB_exprs = [
        parse_expr(s) if isinstance(s, str) else s
        for s in KB
    ]

    alpha_expr = (
        parse_expr(alpha) if isinstance(alpha, str)
        else alpha
    )

    symbols_set = set()

    for sentence in KB_exprs:
        symbols_set.update(sentence.get_symbols())

    symbols_set.update(alpha_expr.get_symbols())
    symbols = sorted(symbols_set)

    entails_result = TT_CHECK_ALL(
        KB_exprs, alpha_expr, symbols, {}
    )

    print("\n" + "=" * 100)
    print("GENERATED TRUTH TABLE")
    print("=" * 100)

    headers = symbols + [
        "KB Sentences",
        "Combined KB",
        "Query (alpha)"
    ]

    print(" | ".join(headers))
    print("-" * 100)

    for row in truth_table_rows:
        model = row["model"]

        values = [
            "T" if model[sym] else "F"
            for sym in symbols
        ]

        kb_status = ", ".join(
            "T" if sentence.evaluate(model) else "F"
            for sentence in KB_exprs
        )

        values.append(f"[{kb_status}]")
        values.append("T" if row["kb"] else "F")
        values.append("T" if row["alpha"] else "F")

        print(" | ".join(values))

    print("=" * 100)

    return entails_result


if __name__ == "__main__":
    print("=" * 60)
    print("NAME : Hitesh        USN : 1BM24CS117")
    print("BMSCE - TRUTH TABLE ENUMERATION GENERATOR")
    print("=" * 60)

    print("Operators:")
    print("~   = NOT")
    print("&   = AND")
    print("|   = OR")
    print(">>  = IMPLICATION")
    print("<<>> = BICONDITIONAL")
    print("-" * 60)

    user_kb = []

    print("Enter Knowledge Base (KB) sentences.")
    print("Press Enter on an empty line when finished.")

    while True:
        sentence = input(
            f"Enter KB sentence #{len(user_kb) + 1}: "
        ).strip()

        if not sentence:
            break

        user_kb.append(sentence)

    if not user_kb:
        print("Error: Knowledge base cannot be empty.")
        raise SystemExit

    user_query = input(
        "Enter the query sentence (alpha): "
    ).strip()

    try:
        entails = TT_ENTAILS(user_kb, user_query)

        print("\n" + "=" * 60)

        if entails:
            print(
                "FINAL RESULT: KB ENTAILS alpha "
                "(KB |= alpha) -> True"
            )
        else:
            print(
                "FINAL RESULT: KB DOES NOT ENTAIL alpha "
                "(KB |/= alpha) -> False"
            )

        print("=" * 60)

    except Exception as e:
        print(f"Runtime Parsing Error: {e}")


NAME : Hitesh        USN : 1BM24CS117
BMSCE - TRUTH TABLE ENUMERATION GENERATOR
Operators:
~   = NOT
&   = AND
|   = OR
>>  = IMPLICATION
<<>> = BICONDITIONAL
------------------------------------------------------------
Enter Knowledge Base (KB) sentences.
Press Enter on an empty line when finished.
Enter KB sentence #1: (A|C)&(B|~C)
Enter KB sentence #2: 
Enter the query sentence (alpha): A|B

GENERATED TRUTH TABLE
A | B | C | KB Sentences | Combined KB | Query (alpha)
----------------------------------------------------------------------------------------------------
F | F | F | [F] | F | F
F | F | T | [F] | F | F
F | T | F | [F] | F | T
F | T | T | [T] | T | T
T | F | F | [T] | T | T
T | F | T | [F] | F | T
T | T | F | [T] | T | T
T | T | T | [T] | T | T

FINAL RESULT: KB ENTAILS alpha (KB |= alpha) -> True
